# 🏥 Hybrid Clinical Decision Support System - Colab Training

This notebook trains a multi-class skin disease classifier using:
- **Melanoma** Detection
- **Eczema** Detection
- **Psoriasis** Detection  
- **Acne** Detection
- **Normal/Healthy Skin** Detection

**Updated Features:**
- Uses unified Kaggle dataset: `mateenzahid/skin-diesease` (~819MB)
- Single download instead of 5 separate datasets
- GPU acceleration for faster training (TPU also supported)
- Automatically detects and uses GPU/TPU/CPU
- Handles datasets with mixed folder structures (uses ALL images)

**Hardware Requirements:**
- GPU: Recommended (Runtime → Change runtime type → GPU - T4, V100, or A100)
- TPU: Also supported (Runtime → Change runtime type → TPU)
- RAM: 12.7 GB High-RAM runtime if available
- Disk: ~1GB for dataset and models

## 📦 Step 1: Setup Environment

In [9]:
# Check hardware availability (GPU/TPU)
import torch
import os

print(f"PyTorch version: {torch.__version__}")

# Check for GPU first (prioritized)
if torch.cuda.is_available():
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = torch.device('cuda')
    use_tpu = False
else:
    # Fall back to TPU if GPU not available
    try:
        import torch_xla
        device = torch_xla.device()
        print(f"✓ TPU detected: {device}")
        print(f"TPU cores available")
        use_tpu = True
    except:
        use_tpu = False
        print("⚠️ No GPU/TPU detected. Training will be slower on CPU.")
        print("Consider: Runtime → Change runtime type → Hardware accelerator → GPU")
        device = torch.device('cpu')

print(f"\nUsing device: {device}")

PyTorch version: 2.10.0+cu128
✓ GPU detected: Tesla T4
GPU Memory: 14.6 GB

Using device: cuda


In [10]:
# Install required packages
!pip install -q timm pyyaml gradio kaggle requests tqdm scikit-learn pillow opencv-python shap

# Install TPU support if using TPU (optional, GPU is prioritized)
try:
    import torch_xla
    print("✓ TPU libraries already installed")
except:
    print("TPU libraries not installed (GPU will be used if available)")
    # Uncomment below if you want to install TPU support:
    # !pip install -q cloud-tpu-client==0.10 torch-xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

TPU libraries not installed (GPU will be used if available)


## 💾 Step 2: Mount Google Drive (Optional - for saving models)

In [11]:
# Mount Google Drive to save trained models
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive
import os
project_dir = '/content/drive/MyDrive/clinical_diagnosis_system'
os.makedirs(project_dir, exist_ok=True)
print(f"✓ Project directory: {project_dir}")

Mounted at /content/drive
✓ Project directory: /content/drive/MyDrive/clinical_diagnosis_system


## 📥 Step 3: Clone Project from GitHub (or upload files)

In [12]:
# Option A: Clone from GitHub (if you have a repo)
# !git clone https://github.com/yourusername/your-repo.git
# %cd your-repo

# Option B: Upload project files manually
# Use Colab's file upload feature to upload your project zip

# Option C: Start fresh (download individual files)
# This notebook assumes you have the project structure

# For this example, we'll create minimal structure
import os
os.makedirs('src', exist_ok=True)
os.makedirs('config', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

print("✓ Directory structure created")

✓ Directory structure created


## 🔑 Step 4: Setup Kaggle API Credentials

In [13]:
# Upload your kaggle.json file
# Download from: https://www.kaggle.com/settings → API → Create New API Token

from google.colab import files
import os

print("📤 Upload your kaggle.json file when prompted...")
uploaded = files.upload()

# Setup Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✓ Kaggle API configured!")

📤 Upload your kaggle.json file when prompted...


Saving kaggle.json to kaggle (1).json
✓ Kaggle API configured!


## 📊 Step 5: Download Datasets

In [14]:
# Download unified skin disease dataset from Kaggle
import kaggle
from pathlib import Path

data_dir = Path('data')
raw_dir = data_dir / 'skin_lesions_raw'
raw_dir.mkdir(parents=True, exist_ok=True)

# Use unified dataset (819MB total with all 5 disease classes including normal skin)
DATASET_ID = 'mateenzahid/skin-diesease'

print("📥 Downloading unified skin disease dataset...")
print(f"Dataset: {DATASET_ID}")
print("This contains melanoma, eczema, psoriasis, acne, and normal skin images")
print("Total size: ~819MB\n")

try:
    kaggle.api.dataset_download_files(
        DATASET_ID,
        path=str(raw_dir),
        unzip=True,
        quiet=False
    )
    print("\n✓ Dataset downloaded successfully!")

    # List downloaded structure
    print("\n📁 Downloaded structure:")
    for item in sorted(raw_dir.iterdir()):
        if item.is_dir():
            img_count = len(list(item.rglob('*.jpg'))) + len(list(item.rglob('*.png')))
            print(f"  - {item.name}/  ({img_count} images)")
except Exception as e:
    print(f"✗ Error downloading dataset: {e}")
    print("Please check your Kaggle API credentials and dataset permissions")

📥 Downloading unified skin disease dataset...
Dataset: mateenzahid/skin-diesease
This contains melanoma, eczema, psoriasis, acne, and normal skin images
Total size: ~819MB

Dataset URL: https://www.kaggle.com/datasets/mateenzahid/skin-diesease


100%|██████████| 748M/748M [00:04<00:00, 168MB/s]




✓ Dataset downloaded successfully!

📁 Downloaded structure:
  - skin_lesions_raw/  (24298 images)


## 🔧 Step 6: Verify Dataset Structure

The unified dataset already contains organized train/val/test splits, so we'll use it directly without reorganization.

In [ ]:
# Detect cleaned dataset root (accepts `data/` or Kaggle upload paths like `data/skin_lesion_raw`)
from pathlib import Path
import zipfile
import os

candidates = [
    Path('data'),
    Path('data/skin_lesion_raw'),
    Path('data/skin_lesions_raw'),
    Path('data/skin_lesion'),
    Path('data/skin_lesions')
]

# If data/ missing, attempt to extract data.zip
if not any(p.exists() for p in candidates):
    zip_path = Path('data.zip')
    if zip_path.exists():
        print('Extracting data.zip to current directory...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('.')
        print('Extraction complete.')
    else:
        print('⚠️ No candidate dataset directories found and data.zip not present.')

# Choose first candidate that contains train/val/test
data_root = None
for cand in candidates:
    if cand.exists() and all((cand / s).exists() for s in ['train','val','test']):
        data_root = cand
        break

# Fallback: prefer top-level `data` if it contains any of the expected folders
if data_root is None and Path('data').exists():
    if any((Path('data')/s).exists() for s in ['train','val','test']):
        data_root = Path('data')

if data_root is None:
    print('⚠️ Could not find a dataset root with train/val/test folders. Checked candidates:')
    for c in candidates:
        print(' -', c)
    data_root = Path('data')  # default to data/ (will show missing folders below)

splits = ['train','val','test']
classes = ['melanoma','eczema','psoriasis','acne','normal']

print('Using dataset root:', data_root)
for s in splits:
    for c in classes:
        p = data_root / s / c
        cnt = 0
        if p.exists():
            for ext in ('*.jpg','*.jpeg','*.png','*.bmp'):
                cnt += len(list(p.rglob(ext)))
        print(f"  - {s}/{c}: {cnt}")

print('\nDataset root selected — the pipeline will use this path for train/val/test.')


📁 Verifying dataset structure...

✓ melanoma:
  - Total images: 10605
✓ eczema:
  - Total images: 3123
✓ psoriasis:
  - Total images: 2801
✓ acne:
  - train/: 2778 images
  - test/: 918 images
  - valid/: 921 images
✓ normal:
  - Total images: 3152

✓ Dataset structure verified!
We'll use the existing folder structure directly (no copying/reorganization needed)


## 🎓 Step 7: Train the CNN Model

In [ ]:
# Load data directly from detected root (no splitting) — respects `data_root` detected above
from pathlib import Path

def prepare_data():
    """Load data from detected `data_root` with splits: train/val/test.
    Returns: train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names
    """
    # use data_root from previous detection cell if available, otherwise default to 'data'
    try:
        root = data_root
    except NameError:
        root = Path('data')

    # Full class names (preserve this order)
    class_names = [
        "Melanoma Skin Cancer Nevi and Moles",
        "Eczema Photos",
        "Psoriasis pictures Lichen Planus and related diseases",
        "Acne and Rosacea Photos",
        "Normal Healthy Skin"
    ]

    # Directory short names (must match folders inside the chosen root)
    short_names = ['melanoma','eczema','psoriasis','acne','normal']
    class_to_idx = {full: i for i, full in enumerate(class_names)}
    short_to_full = dict(zip(short_names, class_names))

    train_paths, train_labels = [], []
    val_paths, val_labels = [], []
    test_paths, test_labels = [], []

    for short in short_names:
        full = short_to_full[short]
        idx = class_to_idx[full]
        for split, paths_list, labels_list in (
            ('train', train_paths, train_labels),
            ('val', val_paths, val_labels),
            ('test', test_paths, test_labels),
        ):
            p = Path(root) / split / short
            if not p.exists():
                print(f"⚠️ Missing folder: {p} (0 files)")
                continue
            for img_path in sorted(p.rglob('*')):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    paths_list.append(str(img_path))
                    labels_list.append(idx)

    # Print simple summary
    for split_name, paths, labels in (('train', train_paths, train_labels), ('val', val_paths, val_labels), ('test', test_paths, test_labels)):
        counts = {name:0 for name in class_names}
        for lab in labels:
            counts[class_names[lab]] += 1
        print(f"\n{split_name} totals (from {root}):")
        total = 0
        for name in class_names:
            print(f"  {name}: {counts[name]}")
            total += counts[name]
        print(f"  -> total {split_name}: {total}")

    return train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names

# Execute to load lists into notebook variables
train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names = prepare_data()

📊 Loading images from dataset...

✓ melanoma: 10605 images total → 8484 train, 2121 val
✓ eczema: 3123 images total → 2498 train, 625 val
✓ psoriasis: 2806 images total → 2244 train, 562 val
✓ acne/train: 2778 images
✓ acne/val: 921 images
✓ normal: 3152 images total → 2521 train, 631 val

📈 Dataset Summary:
Training samples: 18525
Validation samples: 4860
Total images: 23385


In [ ]:
# Minimal dataset class for this notebook (defines SkinLesionDataset)
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

class SkinLesionDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img_path = self.paths[idx]
        label = int(self.labels[idx])
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            # If an image fails to load, return a black image of the expected size
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0,0,0))
        if self.transform is not None:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

In [ ]:
# Define consistent transforms for train/val/test
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SkinLesionDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

IMG_SIZE = 384

common_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Use same transform for train/val/test as requested
train_transform = common_transform
val_transform = common_transform
test_transform = common_transform

# Create datasets and dataloaders
train_dataset = SkinLesionDataset(train_paths, train_labels, train_transform)
val_dataset = SkinLesionDataset(val_paths, val_labels, val_transform)
test_dataset = SkinLesionDataset(test_paths, test_labels, test_transform)

# Adjust batch size based on IMG_SIZE and GPU memory
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {batch_size}")
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

Image size: 384x384
Batch size: 16
Batches per epoch: 1158


In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm

# Device setup
if torch.cuda.is_available():
    device = torch.device('cuda')
    use_tpu = False
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    try:
        import torch_xla
        device = torch_xla.device()
        use_tpu = True
        print(f"✓ Using TPU: {device}")
    except:
        device = torch.device('cpu')
        use_tpu = False
        print(f"⚠️ Using CPU (slower)")

# Model
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5)
model = model.to(device)

# Loss
criterion = nn.CrossEntropyLoss()

# Debug
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✓ Model initialized")

✓ Using GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model parameters: 4,013,953
✓ Model initialized


In [ ]:
# Training loop with TPU/GPU support
from tqdm import tqdm
def train_model(model, train_loader, val_loader, epochs=10):
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    # Check if using TPU
    try:
        import torch_xla
        import torch_xla.core.xla_model as xm
        use_tpu = True
    except:
        use_tpu = False

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            if use_tpu:
                xm.optimizer_step(optimizer)  # TPU-specific optimizer step
            else:
                optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({'loss': train_loss/len(pbar), 'acc': 100.*train_correct/train_total})

        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = 100. * val_correct / val_total

        print(f"\nEpoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc

            # For TPU, move model to CPU before saving
            if use_tpu:
                cpu_model = model.cpu()
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': cpu_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'class_names': class_names
                }, 'models/cnn_skin_lesion.pth')
                model = model.to(device)  # Move back to TPU
            else:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'class_names': class_names
                }, 'models/cnn_skin_lesion.pth')

            print(f"  ✓ Best model saved (Val Acc: {val_acc:.2f}%)")

    return history

# Train for 10 epochs (increase for better results)
history = train_model(model, train_loader, val_loader, epochs=10)

Epoch 1/10: 100%|██████████| 1158/1158 [05:10<00:00,  3.73it/s, loss=0.293, acc=88.1]



Epoch 1:
  Train Loss: 0.2925, Train Acc: 88.15%
  Val Loss: 0.1654, Val Acc: 92.35%
  ✓ Best model saved (Val Acc: 92.35%)


Epoch 2/10: 100%|██████████| 1158/1158 [05:11<00:00,  3.72it/s, loss=0.29, acc=87.8]



Epoch 2:
  Train Loss: 0.2900, Train Acc: 87.82%
  Val Loss: 0.1471, Val Acc: 91.58%


Epoch 3/10:  33%|███▎      | 387/1158 [01:44<03:10,  4.04it/s, loss=0.0637, acc=91.2]

## 📊 Step 8: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy')
ax2.plot(history['val_acc'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\n🎯 Final Results:")
print(f"  Best Validation Accuracy: {max(history['val_acc']):.2f}%")
print(f"  Final Training Accuracy: {history['train_acc'][-1]:.2f}%")

## 💾 Step 9: Save Model to Google Drive

In [ ]:
# Copy trained model to Google Drive
!cp models/cnn_skin_lesion.pth /content/drive/MyDrive/clinical_diagnosis_system/

print("✓ Model saved to Google Drive!")
print("You can now download it and use it locally.")

## 🧪 Step 10: Test the Model

In [ ]:
# Full evaluation on `data/test/` — classification report + confusion matrix
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

# Convert to numpy
y_true = np.array(all_labels)
y_pred = np.array(all_preds)

# Class names (full)
labels = class_names

# Classification report
report = classification_report(y_true, y_pred, target_names=labels, digits=4, output_dict=True)
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=labels, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Save report and confusion matrix numbers
out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
with open(out_dir / 'test_classification_report.json', 'w') as fh:
    json.dump(report, fh, indent=2)
np.save(out_dir / 'test_confusion_matrix.npy', cm)
print(f"Saved report to {out_dir / 'test_classification_report.json'} and confusion matrix to {out_dir / 'test_confusion_matrix.npy'}")


## ✅ Summary

You've successfully:
1. ✓ Set up the environment with GPU support (TPU optional)
2. ✓ Downloaded unified skin disease dataset from Kaggle (mateenzahid/skin-diesease)
3. ✓ Loaded ALL images from dataset (~24,000+ images) with smart split handling
4. ✓ Trained an EfficientNet-B0 model on 5 disease classes (including Normal skin)
5. ✓ Visualized training results
6. ✓ Saved the model to Google Drive

**Dataset Info:**
- Source: https://www.kaggle.com/datasets/mateenzahid/skin-diesease
- Size: ~819MB total (~24,000+ images)
- Classes: Melanoma, Eczema, Psoriasis, Acne, Normal/Healthy Skin
- Structure: Uses ALL images - handles both split and non-split folder structures
  - Diseases with splits (acne): Uses existing train/val/test folders
  - Diseases without splits (melanoma, eczema, psoriasis, normal): Auto-splits 80/20
- All original data preserved in data/skin_lesions_raw/

**Model Performance:**
- The model now correctly identifies normal/healthy skin to avoid false positives
- Uses GPU acceleration for faster training (TPU also supported)
- EfficientNet-B0 backbone with transfer learning
- Trains on ~20,000 images (80% of total dataset)

### Next Steps:
- Download the model from Google Drive (`cnn_skin_lesion.pth`)
- Place it in your local project: `models/cnn_skin_lesion.pth`
- Run the web interface: `python app.py`
- Test with both diseased and healthy skin images

### Tips for Better Results:
- Train for 20-50 epochs (this notebook uses 10 for speed)
- Use learning rate scheduling
- Add test-time augmentation
- Try ensemble methods with multiple models
- Increase batch size if using GPU with sufficient memory (32-64 works well on T4/V100)